# 07 · Final comparison — §3, §4, §6

Builds every comparison table in the report from committed results. In `synthetic` / `debug` mode it previews whatever has landed without writing anything.

| | |
|---|---|
| **Owner** | Member 4 |
| **Reads** | committed JSON only: `results/metrics/**` |
| **Writes (official)** | `results/tables/*.md`, trade-off figures, `report/figures/` |
| **Run it** | after notebook 06 — no GPU needed, runs in seconds |

In [ ]:
# ── Cell 1 · Colab bootstrap ─ does nothing when run locally ───────────────
# Locally the package is already installed (pip install -e .), so this is skipped.
# On Colab it clones the repo, installs it and makes it importable.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os
    import subprocess

    BRANCH = "main"  # <- your branch, to run work that is not merged yet
    REPO_URL = "https://github.com/ThejithaR/EN3150-Assignment-03-CNN.git"
    REPO_DIR = "/content/EN3150-Assignment-03-CNN"

    if not os.path.exists(REPO_DIR):
        env = dict(os.environ)
        try:  # only needed if the repo is private; the token never leaves the environment
            from google.colab import userdata

            token = userdata.get("GH_TOKEN")
            if token:
                env["GH_TOKEN"] = token
        except Exception:
            pass
        helper = "!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
        subprocess.run(
            ["git", "-c", f"credential.helper={helper}", "clone", "--quiet",
             "--branch", BRANCH, REPO_URL, REPO_DIR],
            env=env,
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR], check=True)
    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")  # an editable install is only seen after a restart
    os.chdir(REPO_DIR)
    print(f"Colab ready: {REPO_DIR} @ {BRANCH}")

In [ ]:
# ── Cell 2 · Setup ─────────────────────────────────────────────────────────
%load_ext autoreload
%autoreload 2

# MODE decides how much runs and whether anything is written:
#   "synthetic"  generated data, 2 epochs, seconds         - before EuroSAT exists
#   "debug"      5% of EuroSAT, 2 epochs                   - find bugs step by step
#   "official"   the full run - the ONLY mode that writes  - then Restart & Run All
MODE = "synthetic"

from edgecnn.config import load_config
from edgecnn.contracts import paths, schema
from IPython.display import Markdown

from edgecnn.evaluation.reporting import (
    build_custom_comparison_table,
    build_final_comparison_table,
    build_optimizer_comparison_table,
    check_same_hardware,
    export_report_figures,
    load_all_runs,
    plot_tradeoff_scatter,
)

In [ ]:
# Shared figure style (Member 1): small inline figures, print-quality saved PNGs.
from edgecnn.evaluation.plotting import apply_style

apply_style()

## Write switch

Tables and figures span several runs, so this notebook uses one switch.

In [ ]:
WRITE = MODE == "official"

## 1 · What has landed

In [ ]:
runs = load_all_runs()
sorted(runs)

## 2 · Same hardware within each table?

Epoch times are only comparable within a table if those runs trained on the same runtime.

In [ ]:
tables = load_config("configs/experiments/model_b__adam.yaml").section("reporting")["tables"]
{table["name"]: check_same_hardware(table["runs"]) for table in tables}

## 3 · §3 — optimizer comparison

In [ ]:
Markdown(build_optimizer_comparison_table(write=WRITE))

## 4 · §4 — Model A vs Model B

In [ ]:
Markdown(build_custom_comparison_table(write=WRITE))

## 5 · §6 — Model B vs the SOTA backbones

In [ ]:
Markdown(build_final_comparison_table(write=WRITE))

## 6 · The trade-off in one picture

Parameters track memory; MACs track compute. A model can be cheap in one and expensive in the other.

In [ ]:
plot_tradeoff_scatter("trainable_params", write=WRITE)

In [ ]:
plot_tradeoff_scatter("macs", write=WRITE)

## 7 · Export figures for the report

In [ ]:
export_report_figures(write=WRITE)

## Discussion prompts — §6

- State ratios, not just both numbers: "N× fewer parameters for M points of accuracy".
- The 64×64 handicap on the SOTA backbones — say so, rather than claiming a clean win.
- Pretrained features matter most with scarce data; would the answer flip at 2,700 images?
- Peak activation memory, not parameter count, is often what blocks MCU deployment.
- "It depends on the memory budget" is strong only if you name the threshold where the answer flips.

## Official-run checklist

- [ ] `MODE = "official"` in the setup cell
- [ ] Kernel → **Restart & Run All** finished without errors
- [ ] every output is what you expect — no stray warnings or giant prints
- [ ] no hardware warnings above, or they are explained in the report
- [ ] `pytest tests/contracts` is green
- [ ] commit this notebook **with its outputs**, together with the files it wrote

In [ ]:
# ── Colab only: send this run's results back to GitHub ─────────────────────
# Does nothing locally, or in synthetic / debug mode. Procedure: notebooks/README.md
if IN_COLAB and MODE == "official":
    from edgecnn.utils import push_results

    push_results(
        [paths.TABLES_DIR, paths.FIGURES_DIR, paths.REPORT_FIGURES_DIR],
        "Add comparison tables and figures (Colab)",
    )